In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torch.optim import Adam

In [ ]:
df = pd.read_csv("week5/Day4/stock_data.csv")

df.head()

In [ ]:
df.columns

In [ ]:
df["Target"] = df["Close"].shift(-1)

df.dropna(inplace=True)

In [ ]:
features = ["Open", "High", "Low", "Close", "Volume"]

X = df[features]
y = df["Target"]

In [ ]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)

y_scaled = scaler_y.fit_transform(
    y.values.reshape(-1,1)
)

In [ ]:
sequence_length = 30

X_sequences = []
y_sequences = []

for i in range(len(X_scaled) - sequence_length):
    X_sequences.append(
        X_scaled[i:i+sequence_length]
    )

    y_sequences.append(
        y_scaled[i+sequence_length]
    )

X_sequences = np.array(X_sequences)
y_sequences = np.array(y_sequences)

In [ ]:
train_size = int(0.7 * len(X_sequences))
val_size = int(0.15 * len(X_sequences))

X_train = X_sequences[:train_size]
y_train = y_sequences[:train_size]

X_val = X_sequences[
    train_size:train_size+val_size
]
y_val = y_sequences[
    train_size:train_size+val_size
]

X_test = X_sequences[
    train_size+val_size:
]
y_test = y_sequences[
    train_size+val_size:
]

In [ ]:
class StockDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_dataset = StockDataset(
    X_train,
    y_train
)

val_dataset = StockDataset(
    X_val,
    y_val
)

test_dataset = StockDataset(
    X_test,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
class LSTMModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=5,
            hidden_size=64,
            num_layers=2,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.2)

        self.fc = nn.Linear(
            64,
            1
        )

    def forward(self, x):

        out, _ = self.lstm(x)

        out = out[:, -1, :]

        out = self.dropout(out)

        out = self.fc(out)

        return out

In [ ]:
model = LSTMModel()

criterion = nn.MSELoss()

optimizer = Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
epochs = 20

for epoch in range(epochs):

    model.train()

    train_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(
            outputs,
            y_batch
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs}, "
        f"Loss: {train_loss:.4f}"
    )

In [ ]:
model.eval()

predictions = []
actuals = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        outputs = model(X_batch)

        predictions.extend(
            outputs.numpy()
        )

        actuals.extend(
            y_batch.numpy()
        )

In [ ]:
r2 = r2_score(
    actuals,
    predictions
)

print("R² :", r2)

In [ ]:
torch.save(
    model.state_dict(),
    "lstm_stock_model.pth"
)

In [ ]:
import joblib

joblib.dump(
    scaler_X,
    "stock_scaler.pkl"
)